In [1]:
"""
    创建向量化模型
"""
from langchain_ollama import OllamaEmbeddings
# 创建向量模型,我们今天使用ollama
ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)

In [2]:
"""
    创建langchain包装过的向量数据库
"""
from app.core.config import settings
# 初始化向量数据库客户端
from langchain_milvus import Milvus, BM25BuiltInFunction

vector_store = Milvus(
    # 稠密向量模型
    embedding_function=ollama_embeddings,
    # collection名称
    collection_name="langchain_collection",
    # 做稀疏向量的操作函数
    builtin_function=BM25BuiltInFunction(
        # 指定中文分词
        analyzer_params={"type": "chinese"}
    ),
    # 向量字段命名
    vector_field=["dense", "sparse"],
    # 数据库连接地址
    connection_args={
        "uri": settings.rag.milvus_url,
    },
    # 是否删除旧的collection，避免重复创建
    drop_old=False,
    # 自动主键
    auto_id=True
)

In [8]:
#  通过数据库对象进行向量匹配查询
user_input = '教育漫画是谁写的'

results = vector_store.similarity_search_with_score(
    query=user_input,
    # 匹配3条
    k=3,
    fetch_k=5,
    # langchain默认采用混合检索,选择权重类型和范围
    ranker_type='weighted',
    ranker_params={'weights': [0.5, 0.5]}
)

# 遍历出来的是元组,然后进行解包
for document, score in results:
    print(f'=================文档得分:{score}======================')
    print(f'文档内容:{document.model_dump_json(indent=2)}')

=================文档得分:0.5262233018875122======================
文档内容:{
  "id": null,
  "metadata": {
    "H3": "（七）教育学创立时期代表人物及主要思想",
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "pk": 468850925521564770
  },
  "page_content": "### （七）教育学创立时期代表人物及主要思想  \n**昆体良**：西方第一个专门论述教育问题的教育家。  \n**培根**：在《论科学的价值和发展》中首次把“教育学”作为一门独立的科学确立下来。  \n**夸美纽斯**（教育学之父、教育史上哥白尼）：代表作《大教学论》是教育学成为一门独立学科的标志。提出“泛智教育”、“班级授课制”和“学年制”；把教师比喻为太阳底下最光辉的事业。  \n**康德**：教育学作为一门课程在大学里讲授，始于康德。  \n**赫尔巴特**（传统教育代表、科学教育学之父）：《普通教育学》是教育学作为一门规范、独立的学科正式诞生的标志。提出以伦理学和心理学作为教育学的基础；教学要有教育性；三中心论：教师、教材、课堂；四阶段论：清楚、联想、系统和方法。  \n**杜威**（现代教育代表、实用主义哲学之父、儿童中心主义论、教育无目的论）：《民主主义与教育》提出教育的本质：教育即生活、教育即生长、教育即经验的改组或改造；学校即社会；从做中学；连带学习；三中心论：儿童、经验、活动；五步教学法：设疑—分析—假设—推断—验证。  \n**卢梭**：《爱弥儿》，提倡自然主义教育思想，认为教育的任务应该使儿童归于自然。  \n**洛克**：《教育漫画》，提出白板说，倡导绅士教育。  \n**裴斯泰洛奇**：《林哈德与葛笃德》，教育心理学化。  \n**斯宾塞**：反对思辨，主张用实证方法研究知识价值；生活预备说；科学知识最有价值，制定以科学知识为核心的课程体系。  \n**陶行知**：提出生活教育理论：生活即教育，社会即学校，教学做合一；“捧着一颗心来”（师德）；伟大的人民教育家。  \n**蔡元培**：思想自由，兼容并包；以美育代宗教；学界泰斗，人世楷模。",
  "type": "Doc

In [7]:
"""
    基于rrf的混合检索
"""
# 测试混合检索
result = vector_store.similarity_search_with_score(
    query='教育漫画是谁写的',
    k=3,
    fetch_k=5,     # 初筛时召回的文档的数量
    ranker_type='rrf',
    ranker_params={'k': 60}
)

for document, score in result:
    print(f'=================文档得分:{score}======================')
    print(f'文档内容:{document.model_dump_json(indent=2)}')

=================文档得分:0.03201844170689583======================
文档内容:{
  "id": null,
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（七）教育学创立时期代表人物及主要思想",
    "pk": 468850925521564770
  },
  "page_content": "### （七）教育学创立时期代表人物及主要思想  \n**昆体良**：西方第一个专门论述教育问题的教育家。  \n**培根**：在《论科学的价值和发展》中首次把“教育学”作为一门独立的科学确立下来。  \n**夸美纽斯**（教育学之父、教育史上哥白尼）：代表作《大教学论》是教育学成为一门独立学科的标志。提出“泛智教育”、“班级授课制”和“学年制”；把教师比喻为太阳底下最光辉的事业。  \n**康德**：教育学作为一门课程在大学里讲授，始于康德。  \n**赫尔巴特**（传统教育代表、科学教育学之父）：《普通教育学》是教育学作为一门规范、独立的学科正式诞生的标志。提出以伦理学和心理学作为教育学的基础；教学要有教育性；三中心论：教师、教材、课堂；四阶段论：清楚、联想、系统和方法。  \n**杜威**（现代教育代表、实用主义哲学之父、儿童中心主义论、教育无目的论）：《民主主义与教育》提出教育的本质：教育即生活、教育即生长、教育即经验的改组或改造；学校即社会；从做中学；连带学习；三中心论：儿童、经验、活动；五步教学法：设疑—分析—假设—推断—验证。  \n**卢梭**：《爱弥儿》，提倡自然主义教育思想，认为教育的任务应该使儿童归于自然。  \n**洛克**：《教育漫画》，提出白板说，倡导绅士教育。  \n**裴斯泰洛奇**：《林哈德与葛笃德》，教育心理学化。  \n**斯宾塞**：反对思辨，主张用实证方法研究知识价值；生活预备说；科学知识最有价值，制定以科学知识为核心的课程体系。  \n**陶行知**：提出生活教育理论：生活即教育，社会即学校，教学做合一；“捧着一颗心来”（师德）；伟大的人民教育家。  \n**蔡元培**：思想自由，兼容并包；以美育代宗教；学界泰斗，人世楷模。",
  "type": "Do

In [6]:
# 测试混合检索
result = vector_store.similarity_search_with_score(
    query='教育的负面作用是什么',
    k=5,
    fetch_k=5,
    ranker_type='rrf',
    ranker_params={'k': 60},
    expr='H2 like "%第一节%"'  # 过滤条件
)

for document, score in result:
    print(f'=================文档得分:{score}======================')
    print(f'文档内容:{document.model_dump_json(indent=2)}')

=================文档得分:0.032522473484277725======================
文档内容:{
  "id": null,
  "metadata": {
    "H1": "第二章 教育基本原理",
    "H2": "第一节 教育的功能",
    "H3": "（一）个体发展功能和社会发展功能",
    "pk": 468850925521564778
  },
  "page_content": "# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。",
  "type": "Document"
}
=================文档得分:0.03201844170689583======================
文档内容:{
  "id": null,
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（五）荀子主要思想",
    "pk": 468850925521564768
  },
  "page_content": "### （五）荀子主要思想  \n**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  \n**教学原则**：学以致用，锲而不舍。",
  "type": "Document"
}
=================文档得分:0.0317540317773819======================
文档内容:{
  "id": null,
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（三）孔子及《论语》主要思想",
    "pk": 468850925521564766
  